In [1]:
import pandas as pd

from testing.r_and_d.boolean_queries.query_tester import (
    BooleanQueryTester,
    use_baseline_query,
    generate_with_llm,
    query_openalex_minimal,
    reference_df,
)

from testing.r_and_d.boolean_queries.prompts import (
    policy_atlas_v1,
    policy_atlas_v2,
    wang_et_al_q2_prompt,
    wang_et_al_q3_prompt,
)

from app.services.openalex import sanitize_openalex_query
from testing import TESTING_DIR, logger

BOOL_DIR = TESTING_DIR / "r_and_d/boolean_queries/outputs/"
baseline_results = pd.read_json(BOOL_DIR / "baseline_results.jsonl", lines=True)


2025-12-01 14:59:22,823 - app.core.config - INFO - BACKEND_CORS_ORIGINS parsed from comma-separated: ['discoverypolicyatlas-production.up.railway.app', 'http://localhost:3000']


In [ ]:
q = reference_df.question.iloc[3]
q

In [ ]:
query = await use_baseline_query(q, "model", 0, "prompt")
query = sanitize_openalex_query(query)
query

In [ ]:
await query_openalex_minimal(query, count_only=True)

## Prompting experiments

In [ ]:
test_prompt = """
Transform user input into a high quality boolean query 
for querying the OpenAlex academic research database.

# Guidance from OpenAlex
Imagine you are an expert systematic review information specialist; now you are given a systematic review
research topic, with the topic title provided by the user below. Your task is to generate a highly effective systematic
review Boolean query to search on OpenAlex (refer to the professionally made ones); the query needs to be
as inclusive as possible so that it can retrieve all the relevant studies that can be included in the research
topic; on the other hand, the query needs to retrieve fewer irrelevant studies so that researchers can spend
less time judging the retrieved documents.

# Important instructions

Return ONLY the boolean query string, nothing else.
"""

In [ ]:
query = await generate_with_llm(
    question=q,
    model="gpt-5",
    temperature=1.0,
    system_prompt=wang_et_al_q3_prompt,
)
logger.info(q)
logger.info("=" * 20)
logger.info(query)
logger.info("=" * 20)
n_results = await query_openalex_minimal(query, count_only=True)
logger.info(f"Number of retrieved results: {n_results}")

In [ ]:
async def repeated_queries(
    question: str,
    model: str,
    temperature: float,
    system_prompt: str,
    n_runs: int = 5,
) -> str:
    """Generate a boolean query using the given parameters."""
    dfs = []
    for _ in range(n_runs):
        query = await generate_with_llm(
            question=question,
            model=model,
            temperature=temperature,
            system_prompt=system_prompt,
        )
        df, _ = await query_openalex_minimal(query, count_only=False)
        dfs.append(df.assign(query=query))
    return (
        pd.concat(dfs)
        .drop_duplicates(subset=["id"])
    )
    

In [ ]:
q = "what is the impact of financial incentive on parental engagement in evidence based parenting programmes?"

In [ ]:
results = await repeated_queries(q, "gpt-5-mini", 1.0, wang_et_al_q3_prompt, n_runs=3)
len(results)

In [ ]:
results

In [ ]:
pd.set_option("display.max_colwidth", 200)
results.sort_values("relevance_score", ascending=False)[["title", "relevance_score", "query"]]

## Test with new inputs for search wizard


In [2]:
# Import ReferencesService for context-aware query generation
from app.services.analysis.references import ReferencesService

# Initialize the service
references_service = ReferencesService()


In [3]:
# Set up test context
test_research_question = "What interventions improve home learning environment for young children?"
test_population = ["Children under 5", "Low-income families"]
test_outcome = ["Better educational outcomes", "School readiness"]
test_screening_factors = []

logger.info("Test Research Question: %s", test_research_question)
logger.info("Population: %s", test_population)
logger.info("Outcome: %s", test_outcome)
logger.info("Screening Factors: %s", test_screening_factors)


2025-12-01 14:59:29,968 - testing - INFO - Test Research Question: What interventions improve home learning environment for young children?
2025-12-01 14:59:29,969 - testing - INFO - Population: ['Children under 5', 'Low-income families']
2025-12-01 14:59:29,969 - testing - INFO - Outcome: ['Better educational outcomes', 'School readiness']
2025-12-01 14:59:29,970 - testing - INFO - Screening Factors: []


In [13]:
q = """
What interventions improve home learning environment for young children under 5 from low-income families, 
that result in better educational outcomes and higher scool readiness?
"""

query = await generate_with_llm(
    question=q,
    model="gpt-4.1",
    temperature=1.0,
    system_prompt=wang_et_al_q3_prompt,
)
logger.info(q)
logger.info("=" * 20)
logger.info(query)
logger.info("=" * 20)
n_results = await query_openalex_minimal(query, count_only=True)
logger.info(f"Number of retrieved results: {n_results}")

2025-12-01 15:18:24,797 - testing - INFO - 
What interventions improve home learning environment for young children under 5 from low-income families, 
that result in better educational outcomes and higher scool readiness?

2025-12-01 15:18:24,798 - testing - INFO - ====================
2025-12-01 15:18:24,798 - testing - INFO - ((("home learning environment" OR "home-based learning" OR "family learning environment" OR "home education environment" OR "home literacy environment" OR "home numeracy environment" OR "home educational environment" OR "home stimulation" OR "home activities" OR "family environment" OR "parent-child interaction" OR "parental engagement" OR "parental involvement" OR "parenting practices" OR "parent support" OR "parental support" OR "parent-child relationship" OR "early childhood stimulation" OR "home learning activities") AND ("intervention*" OR "program*" OR "initiative*" OR "training" OR "support" OR "education" OR "workshop*" OR "home visit*" OR "parenting pro

In [6]:
# Test single boolean query generation from context
query_from_context = await (
    references_service.generate_boolean_query_from_context(
        research_question=test_research_question,
        population_selected=test_population,
        outcome_selected=test_outcome,
        screening_factors=test_screening_factors,
    )
)

logger.info("=" * 50)

# Test query on OpenAlex
query_sanitized = sanitize_openalex_query(query_from_context)
n_results = await query_openalex_minimal(query_sanitized, count_only=True)
logger.info(f"Number of retrieved results: {n_results}")

2025-12-01 14:59:50,045 - app.services.analysis.references - INFO - 🔍 Generating boolean query from search context
2025-12-01 14:59:55,172 - app.services.analysis.references - INFO - ✅ Generated boolean query from context: '("home learning environment" OR "home literacy environment" OR "home numeracy environment" OR "home-based learning" OR "family learning environment" OR "learning at home" OR "parent-child interaction" OR "parental involvement" OR "parent-child reading" OR "parent-child activities" OR "home education" OR "home instruction") AND (intervention* OR program* OR training OR education* OR support OR strategy OR curriculum* OR initiative* OR "parenting program*" OR "family intervention*") AND (child* OR toddler* OR infant* OR "preschool child*" OR "children under 5" OR "early childhood" OR "early years" OR "pre-kindergarten" OR "pre-kindergarten children" OR "preschooler*") AND ("low income" OR disadvantaged OR "socioeconomic disadvantage" OR "low-income families" OR "econo

In [11]:
# Test multi-query generation from context
queries_multi = await references_service.generate_boolean_queries_multi_from_context(
    research_question=test_research_question,
    population_selected=test_population,
    outcome_selected=test_outcome,
    screening_factors=test_screening_factors,
    n_runs=3,
    temperature=1.0,
)

logger.info("=" * 50)
logger.info("Generated %d Boolean Queries from Context:", len(queries_multi))
logger.info("=" * 50)
for i, q in enumerate(queries_multi, 1):
    logger.info(f"\nQuery {i}:")
    logger.info(q)
logger.info("=" * 50)


2025-12-01 15:17:29,989 - app.services.analysis.references - INFO - 🔍 Generating 3 boolean queries from context (temp=1.0)
2025-12-01 15:17:49,879 - app.services.analysis.references - INFO - ✅ Generated 3 queries (3 unique) from context for multi-query search
2025-12-01 15:17:49,880 - testing - INFO - ==================================================
2025-12-01 15:17:49,880 - testing - INFO - Generated 3 Boolean Queries from Context:
2025-12-01 15:17:49,880 - testing - INFO - ==================================================
2025-12-01 15:17:49,881 - testing - INFO - 
Query 1:
2025-12-01 15:17:49,881 - testing - INFO - (("home learning environment" OR "home-based learning" OR "learning at home" OR "家庭学习环境" OR "家庭教育环境" OR "parent-child interaction" OR "family literacy" OR "parental engagement" OR "parental involvement" OR "parent-child reading" OR "early literacy activities" OR "shared reading" OR "home numeracy" OR "home literacy") 
AND 
(intervention* OR program* OR strateg* OR init

In [12]:
# Test multi-query retrieval - combine results from all queries
dfs_multi = []
for i, query in enumerate(queries_multi):
    query_sanitized = sanitize_openalex_query(query)
    df, _ = await query_openalex_minimal(query_sanitized, count_only=False)
    if df is not None and len(df) > 0:
        dfs_multi.append(df.assign(query_num=i+1, query=query))

if dfs_multi:
    combined_results = pd.concat(dfs_multi).drop_duplicates(subset=["id"])
    logger.info(f"Total unique results from multi-query: {len(combined_results)}")
    logger.info(f"Results per query: {[len(df) for df in dfs_multi]}")
else:
    logger.info("No results retrieved from multi-query")


2025-12-01 15:17:54,040 - testing - INFO - Total unique results from multi-query: 248
2025-12-01 15:17:54,041 - testing - INFO - Results per query: [142, 54, 149]
